# DART LLM Fine-tuning
EXAONE-3.5-2.4B-Instruct + QLoRA (4-bit)
RTX 4070 Ti (12GB VRAM) 기준

In [1]:
import json
import torch
from pathlib import Path
from trl import SFTTrainer
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: NVIDIA GeForce RTX 4070 Ti
VRAM: 12.9 GB


## 1. 설정

In [2]:
MODEL_ID   = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
DATASET    = '../data/processed/dart_qa_dataset.jsonl'
LORA_OUT   = '../outputs/lora'
MERGED_OUT = '../outputs/merged'


LORA_R       = 16
LORA_ALPHA   = 32
BATCH_SIZE   = 4
GRAD_ACCUM   = 4    
MAX_SEQ_LEN  = 512
EPOCHS       = 3
LR           = 2e-4

## 2. 데이터셋 로드 및 포맷

In [3]:
# JSONL 로드
records = []
with open(DATASET, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

print(f'총 {len(records)}건')

# EXAONE 채팅 포맷으로 변환
SYSTEM = '당신은 한국 금융 공시(DART) 전문 AI 어시스턴트입니다. 공시 본문을 바탕으로 핵심 내용을 정확하고 간결하게 답변합니다.'

def format_prompt(rec):
    user = f'[공시 본문]\n{rec["input"][:1500]}\n\n[질문]\n{rec["instruction"]}' if rec.get('input') else rec['instruction']
    return {
        'text': (
            f'[|system|]{SYSTEM}[|endofturn|]'
            f'[|user|]{user}[|endofturn|]'
            f'[|assistant|]{rec["output"]}[|endofturn|]'
        )
    }

data = Dataset.from_list([format_prompt(r) for r in records])
data = data.train_test_split(test_size=0.1, seed=42)
print(f'train: {len(data["train"])} / eval: {len(data["test"])}')

총 14277건
train: 12849 / eval: 1428


## 3. 모델 로드 (4-bit QLoRA)

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('모델 로드 완료')

[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in C:\Users\vacke\.cache\huggingface\modules\transformers_modules\LGAI_hyphen_EXAONE\EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct\ccce25bd39c141fe053e0bc75818a8f5fe962802\modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in C:\Users\vacke\.cache\huggingface\modules\transformers_modules\LGAI_hyphen_EXAONE\EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct\ccce25bd39c141fe053e0bc75818a8f5fe962802\modeling_exaone.py.


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\vacke\anaconda3\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\vacke\anaconda3\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\vacke\anaconda3\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\vacke\anaconda3\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] ExaoneForCausalLM does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


모델 로드 완료


## 4. LoRA 설정

In [5]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

inner_cls = type(model.transformer)
inner_cls.get_input_embeddings = lambda self: self.wte
inner_cls.set_input_embeddings = lambda self, v: setattr(self, 'wte', v)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,529,600 || all params: 2,410,856,960 || trainable%: 0.2294


## 5. 학습

In [6]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=LORA_OUT,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=False,
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
    max_length=MAX_SEQ_LEN,       
    dataset_text_field='text',         
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=data['train'],
    eval_dataset=data['test'],
    args=training_args,
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/12849 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/12849 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1428 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1428 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 361}.


Step,Training Loss,Validation Loss
50,2.098050,1.890772
100,0.974655,0.925233
150,0.737488,0.734500
200,0.632416,0.670197
250,0.675465,0.637007
300,0.697448,0.606032
350,0.537580,0.585599
400,0.606188,0.569961
450,0.517766,0.556428
500,0.569151,0.543636


TrainOutput(global_step=2412, training_loss=0.5535862324249685, metrics={'train_runtime': 23515.0818, 'train_samples_per_second': 1.639, 'train_steps_per_second': 0.103, 'total_flos': 2.5204573198866432e+17, 'train_loss': 0.5535862324249685})

## 6. LoRA 병합 → 전체 모델 저장

In [10]:
from peft import PeftModel

# 4-bit 모델은 병합 전 fp16으로 재로드 필요
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_model, LORA_OUT)
merged = merged.merge_and_unload()

merged.save_pretrained(MERGED_OUT)
tokenizer.save_pretrained(MERGED_OUT)
print(f'병합 완료: {MERGED_OUT}')

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

병합 완료: ../outputs/merged


## 7. 추론 테스트

In [18]:
test_input = '이 사업보고서에서 회사의 주요 사업 내용을 요약해줘.'
test_context = records[2393]['input'][:1500]

messages = [
    {'role': 'system', 'content': SYSTEM},
    {'role': 'user', 'content': f'[공시 본문]\n{test_context}\n\n[질문]\n{test_input}'},
]

inputs = tokenizer.apply_chat_template(
    messages, return_tensors='pt', add_generation_prompt=True
)

if isinstance(inputs, dict) or hasattr(inputs, 'input_ids'):
    input_ids = inputs['input_ids'].to('cuda')
else:
    input_ids = inputs.to('cuda')

with torch.no_grad():
    outputs = merged.generate(input_ids, max_new_tokens=256, temperature=0.3, do_sample=True)

print(tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True))

이 사업보고서의 'II. 사업의 내용' 1부 '가. 사업의 개요' 부분에 따르면, 주식회사대백저축은행의 주요 사업 내용은 다음과 같습니다.

* **저축은행 업무:** 예금 및 대출 업무를 주요 사업으로 영위하고 있습니다.
* **부수업무:** 저축은행법 및 관련 법령에 따라 허용된 부대업무를 수행하고 있습니다. 구체적인 내용은 동 공시서류의 '사업의 내용'을 참고하시기 바랍니다.
